In [ ]:
# Cell 0 — install + imports
!pip install -q vllm datasets transformers peft anthropic

import json
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

V4_MODEL = "/content/drive/MyDrive/sft/sft-v4-merged-for-eval"
BASE_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct"
TRAIN_JSONL = "/content/drive/MyDrive/sft/train_dataset_clean.jsonl"  # for repo-overlap
ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY") or input("Anthropic API key: ")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_KEY

# Verify v4 shards aren't 0-bytes (Drive corruption lesson)
shards = list(Path(V4_MODEL).glob("*.safetensors"))
assert shards, f"No .safetensors shards found at {V4_MODEL} — wrong path or Drive not mounted?"
for shard in shards:
    size_mb = shard.stat().st_size / 1e6
    print(f"  {shard.name}: {size_mb:.1f} MB")
    assert size_mb > 1.0, f"Shard {shard.name} too small — Drive corruption?"

In [ ]:
# Cell 1 — download SWE-CARE, filter overlap
!python swecare_loader.py --train-jsonl {TRAIN_JSONL} --output ood_input.jsonl

!if [ -s ood_input.jsonl ]; then head -1 ood_input.jsonl | python -c "import json, sys; print(json.dumps(json.loads(sys.stdin.read()), indent=2)[:500])"; else echo "(ood_input.jsonl is empty — check repo-overlap filter or --dry-run setting)"; fi
!wc -l ood_input.jsonl

In [ ]:
# Cell 2 — v4 + base inference (~6h sequential on A100 80GB)
!python run_ood_eval.py \
    --input ood_input.jsonl \
    --output ood_preds.jsonl \
    --v4-model {V4_MODEL} \
    --base-model {BASE_MODEL}

!wc -l ood_preds.jsonl

In [ ]:
# Cell 3 — compute all 6 metrics + 3-vote pairwise
!python ood_metrics.py \
    --preds ood_preds.jsonl \
    --labels ood_input.jsonl \
    --output ood_eval_results.json \
    --api-key {ANTHROPIC_KEY}

import json
results = json.load(open('ood_eval_results.json'))
for k, v in results.items():
    if isinstance(v, dict):
        print(f"{k}:")
        for kk, vv in v.items():
            print(f"  {kk}: {vv:.3f}" if isinstance(vv, float) else f"  {kk}: {vv}")
    else:
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
# Cell 4 — visualizations
import json
import matplotlib.pyplot as plt

results = json.load(open('ood_eval_results.json'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: per-difficulty
diffs = results['iou_lenient_by_difficulty']
axes[0].bar(diffs.keys(), diffs.values())
axes[0].set_title('IoU (lenient) by difficulty')
axes[0].set_ylim(0, 1)

# Plot 2: per-problem-domain
doms = results['iou_lenient_by_problem_domain']
axes[1].barh(list(doms.keys()), list(doms.values()))
axes[1].set_title('IoU (lenient) by problem_domain')
axes[1].set_xlim(0, 1)

# Plot 3: aggregate vs ID baseline (v4 ID was 0.18 ROUGE-L; not directly comparable
# but plot the headline OOD numbers)
metrics = ['iou_strict_mean', 'iou_lenient_mean', 'hit_rate_mean', 'hallucination_rate_mean']
axes[2].bar(metrics, [results.get(m, 0) for m in metrics])
axes[2].set_title('Aggregate metrics')
axes[2].tick_params(axis='x', rotation=45)
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('ood_eval_plots.png', dpi=120)
plt.show()

In [ ]:
# Cell 5 — decision-gate readout per spec
import json
r = json.load(open('ood_eval_results.json'))

print("=" * 60)
print("PHASE 1 DECISION GATE")
print("=" * 60)

pairwise = r.get('pairwise', {}).get('win_rate', 0) * 100
iou_lenient = r['iou_lenient_mean'] * 100
hit = r['hit_rate_mean'] * 100
halluc = r['hallucination_rate_mean'] * 100

print(f"v4 OOD pairwise win:    {pairwise:.1f}%")
print(f"v4 OOD IoU (lenient):   {iou_lenient:.1f}%")
print(f"v4 OOD hit-rate:        {hit:.1f}%")
print(f"v4 OOD hallucination:   {halluc:.1f}%")
print()

# Per-bucket dispersion — if max-min > 30 pts, that's "concentrated weakness"
domain_vals = list(r['iou_lenient_by_problem_domain'].values())
domain_spread = (max(domain_vals) - min(domain_vals)) * 100 if domain_vals else 0
print(f"per-domain IoU spread:  {domain_spread:.1f} pts")
print()

# Spec decision-gate
if pairwise >= 65 and domain_spread < 30:
    branch = "2C — ship v4 as-is, OOD-aware inference hardening"
elif pairwise >= 65 and domain_spread >= 30:
    branch = "2B — targeted fix on the weak domain/difficulty bucket"
elif pairwise >= 50:
    branch = "2A — train more (MelcotCR / dev-mix / hard-case retrain)"
else:
    branch = "Phase 1 review — v4 may need fundamental rework, not Phase 2"

print(f"Recommended Phase 2 branch: {branch}")
print()

# CoRPO viability check (orthogonal to A/B/C)
if iou_lenient >= 30 or hit >= 25:
    print("CoRPO (Phase 2D) is viable — verifiable correctness signal exists")
else:
    print("CoRPO (Phase 2D) not yet viable — IoU/hit-rate too low to form reward threshold")